# Conflict Arbitration Agent — Colab Training Notebook
End to end Colab training for the Conflict Arbitration Agent.
Hardware: T4 GPU.
Environment: https://huggingface.co/spaces/testingaccc/conflict-arbitration-env

In [ ]:
!git clone https://huggingface.co/spaces/testingaccc/conflict-arbitration-env
%cd conflict-arbitration-env
!pip install -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, '.')
from training.grpo_trainer import load_model, get_grpo_config, MODEL_NAME
from training.rollout import collect_rollout
from training.curriculum import CurriculumManager
from training.metrics import TrainingMetrics
from eval.frozen_baseline import save_frozen_checkpoint
from server.client import EnvClient

In [ ]:
model, tokenizer = load_model(MODEL_NAME)
save_frozen_checkpoint(model, tokenizer, './frozen_baseline')

In [ ]:
config = get_grpo_config('./checkpoints')
curriculum = CurriculumManager()
metrics = TrainingMetrics()
env_client = EnvClient('https://testingaccc-conflict-arbitration-env.hf.space')

# Sanity check the live environment
print(env_client.health())

for step in range(2000):
    trajectories = collect_rollout(model, tokenizer, env_client, num_episodes=8)
    metrics.log(step, trajectories, curriculum.current_phase)
    if step % 50 == 0:
        metrics.plot('./training_curves.png')
        print(f'step={step} avg_reward={metrics.history["avg_reward"][-1]:.2f} acc={metrics.history["arbitration_accuracy"][-1]:.2f}')

In [ ]:
model.save_pretrained_merged(
    'conflict-arbitrator-trained',
    tokenizer,
    save_method='merged_16bit',
)